# WildChat In-Depth Insights Analysis

Exploratory analysis of topics, commercial content (brands, intent), themes, and topic modeling for the [WildChat](https://huggingface.co/datasets/allenai/WildChat) dataset. Uses `insights_utils` for logic; optional OpenAI API for purpose/theme/brand labeling (set `USE_OPENAI = True` and add `OPENAI_API_KEY` to `.env`). Outputs are saved under `insights_output/` for reruns.

In [ ]:
# Configuration (edit and run first)
SAMPLE_N = 50           # number of conversations; set to None to use SAMPLE_PCT
SAMPLE_PCT = None         # e.g. 0.01; used only when SAMPLE_N is None
LOAD_FROM_SAVED = False   # if True, load saved tables from INSIGHTS_OUTPUT_DIR where possible
INSIGHTS_OUTPUT_DIR = "insights_output"
USE_OPENAI = True        # set True to run purpose/theme/brand labeling (requires OPENAI_API_KEY in .env)
RANDOM_SEED = 42
N_TOPICS = 15             # for topic modeling (NMF/LDA)

In [ ]:
from pathlib import Path
import pandas as pd
from datasets import load_dataset
from dotenv import load_dotenv

from eda_utils import sample_by_conversation
from insights_utils import (
    extract_text_column,
    add_commercial_intent_heuristic,
    ensure_output_dir,
    load_or_build,
    run_nmf,
    run_lda,
    get_top_terms_nmf,
    get_top_terms_lda,
    assign_dominant_topic,
    build_topic_distribution_df,
    theme_counts_over_time,
    underserved_metrics,
    label_purpose_openai,
    extract_brands_openai,
    label_theme_openai,
    label_commercial_intent_openai,
)
load_dotenv()

In [ ]:
# Check if OpenAI API key is set and works (optional; run before using USE_OPENAI=True)
import os
key = os.environ.get("OPENAI_API_KEY")
if not key or not key.strip():
    print("OPENAI_API_KEY is not set (add it to .env or environment). USE_OPENAI will not work.")
else:
    try:
        from openai import OpenAI
        client = OpenAI()
        r = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": "Say OK"}], max_tokens=5)
        reply = (r.choices[0].message.content or "").strip()
        print("OpenAI API key works. Reply:", reply)
    except Exception as e:
        print("OpenAI API key check failed:", e)

In [ ]:
# Load dataset and sample by conversation (whole conversations only)
dataset = load_dataset("allenai/WildChat", split="train")
sampled = sample_by_conversation(dataset, n=SAMPLE_N, pct=SAMPLE_PCT, seed=RANDOM_SEED)
df = sampled.to_pandas()
df["first_user_text"] = extract_text_column(df, mode="first_user")
print(f"Sampled {len(df)} conversations. Columns: {list(df.columns)}")
df.head(2)

## 1. Topic / industry and commercial content

Commercial intent (keyword heuristic) and optional OpenAI-based purpose, brand extraction, and commercial intent labeling.

In [ ]:
# Commercial intent: keyword heuristic (no API)
df = add_commercial_intent_heuristic(df, text_col="first_user_text")
print("Commercial intent (heuristic):", df["has_commercial_intent"].value_counts())

if USE_OPENAI:
    cache_dir = Path(INSIGHTS_OUTPUT_DIR)
    ensure_output_dir(cache_dir)
    df = label_purpose_openai(df, cache_path=cache_dir / "purpose_labels.parquet", use_cache=LOAD_FROM_SAVED)
    df = extract_brands_openai(df, cache_path=cache_dir / "brands.parquet", use_cache=LOAD_FROM_SAVED)
    df = label_commercial_intent_openai(df, cache_path=cache_dir / "commercial_intent.parquet", use_cache=LOAD_FROM_SAVED)
else:
    print("USE_OPENAI is False; skipping purpose/brand/LLM commercial intent. Set USE_OPENAI=True and OPENAI_API_KEY in .env to enable.")

In [ ]:
# Purpose distribution (if OpenAI was used)
if USE_OPENAI and "purpose" in df.columns:
    display(df["purpose"].value_counts().to_frame("count"))
# Brand frequency (if OpenAI was used)
if USE_OPENAI and "brands" in df.columns:
    import json
    all_brands = []
    for b in df["brands"].dropna():
        if isinstance(b, list):
            all_brands.extend(b)
        elif isinstance(b, str) and b.strip().startswith("["):
            try:
                all_brands.extend(json.loads(b))
            except Exception:
                pass
    if all_brands:
        brand_freq = pd.Series(all_brands).value_counts().head(30)
        display(brand_freq.to_frame("conversation_count"))
# Commercial with vs without brand
if USE_OPENAI and "brands" in df.columns:
    has_brand = df["brands"].map(lambda x: isinstance(x, list) and len(x) > 0 if isinstance(x, list) else False)
    commercial_heuristic = df["has_commercial_intent"]
    commercial_llm = df.get("commercial_intent_llm", pd.Series("No", index=df.index)) == "Yes"
    overlap = pd.crosstab(commercial_heuristic, has_brand, margins=True)
    print("Commercial (heuristic) vs has brand mention:")
    display(overlap)

## 2. Intent / semantic analysis (themes)

Theme labels (OpenAI) and trends: volume over time and underserved metrics.

In [ ]:
if USE_OPENAI:
    cache_dir = Path(INSIGHTS_OUTPUT_DIR)
    df = label_theme_openai(df, cache_path=cache_dir / "theme_labels.parquet", use_cache=LOAD_FROM_SAVED)

if "theme" in df.columns:
    print("Theme distribution:")
    display(df["theme"].value_counts().head(20).to_frame("count"))
else:
    print("No theme column (run with USE_OPENAI=True to add themes).")

In [ ]:
# Theme volume over time and underserved metrics (if theme available)
if "theme" in df.columns:
    theme_over_time = theme_counts_over_time(df, label_col="theme", timestamp_col="timestamp", freq="W")
    print("Sample: theme counts by week (first 20 rows):")
    display(theme_over_time.head(20))
    underserved = underserved_metrics(df, label_col="theme", turn_col="turn")
    print("Underserved (conv_share_pct vs turn_share_pct):")
    display(underserved.sort_values("conv_share_pct"))

## 3. Topic modeling (NMF)

Landscape of topics from first user message (TF-IDF + NMF). Optionally load saved topic assignments.

In [ ]:
# Topic modeling on first user message (no API)
topic_path = Path(INSIGHTS_OUTPUT_DIR) / "topic_assignments.parquet"
texts = df["first_user_text"].fillna("").astype(str)
if LOAD_FROM_SAVED and topic_path.exists():
    topic_df = pd.read_parquet(topic_path)
    df = df.merge(topic_df[["conversation_id", "topic_id"]], on="conversation_id", how="left")
    print("Loaded topic assignments from", topic_path)
else:
    nmf, doc_topics, vectorizer = run_nmf(texts, n_topics=N_TOPICS, random_state=RANDOM_SEED)
    df["topic_id"] = assign_dominant_topic(doc_topics)
    ensure_output_dir(Path(INSIGHTS_OUTPUT_DIR))
    pd.DataFrame({"conversation_id": df["conversation_id"], "topic_id": df["topic_id"]}).to_parquet(topic_path, index=False)
    print("Topic distribution:")
    display(build_topic_distribution_df(doc_topics))
    top_terms = get_top_terms_nmf(nmf, vectorizer, n=10)
    print("Top terms per topic (NMF):")
    for tid, terms in list(top_terms.items())[:10]:
        print(f"  Topic {tid}: {', '.join(terms)}")

In [ ]:
# Topic distribution (counts and share)
if "topic_id" in df.columns:
    dist = df["topic_id"].value_counts().sort_index().reset_index()
    dist.columns = ["topic_id", "count"]
    dist["pct"] = 100.0 * dist["count"] / len(df)
    display(dist)